# Skenario 01 — Build Model Final

Notebook ini membaca hasil ABC dan PSO, membandingkan empat kandidat berdasarkan MSE cross-validation, melatih ulang seluruh kandidat pada semua data `Critical`, dan menyimpan satu model terbaik. Jalankan notebook ABC dan PSO terlebih dahulu.

In [ ]:
from pathlib import Path
import json
import shutil

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

SEED = 42
SENSOR_COLS = [
    "Exhaust Gas Temperature LF (Left Front)",
    "Exhaust Gas Temperature LR (Left Rear)",
    "Exhaust Gas Temperature RF (Right Front)",
    "Exhaust Gas Temperature RR (Right Rear)",
    "Blowby Pressure (Kpa)",
    "Boost Pressure (Kpa)",
    "Engine Oil Temp (°C)",
    "Coolant Temp (°C)",
]
LIFE_COL = "Unit Lifetime (Hour)"
STATUS_COL = "Status"
TARGET_COL = "sisa jam"
FEATURE_COLS = [*SENSOR_COLS, LIFE_COL]
DATA_FILE = "datas/Data_PT.Amanah_Critical_Break.xlsx"
MAX_RUL = 2550.0

## Muat data dan hasil optimasi

In [ ]:
def find_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / DATA_FILE).exists():
            return path
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_FILE}")

def prepare_data(root):
    raw = pd.read_excel(root / DATA_FILE)
    data = raw[[*FEATURE_COLS, STATUS_COL]].copy()
    for col in FEATURE_COLS:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    data[STATUS_COL] = data[STATUS_COL].astype("string").str.strip().str.casefold()
    break_hour = data.loc[data[STATUS_COL].eq("break"), LIFE_COL].max()
    if pd.isna(break_hour):
        raise ValueError("Baris berstatus Break diperlukan untuk membentuk target sisa jam")
    data[TARGET_COL] = (break_hour - data[LIFE_COL]).clip(lower=0, upper=MAX_RUL)
    data = data.loc[data[STATUS_COL].eq("critical")].dropna(subset=[*FEATURE_COLS, TARGET_COL]).copy()
    return data, float(break_hour)

root = find_root()
scenario_dir = root / "newcode/scenarios/01_forecast_sisa_jam_critical"
artifact_dir = scenario_dir / "artifacts"
model_dir = scenario_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)
data, break_hour = prepare_data(root)
candidates = {}
for optimizer in ["abc", "pso"]:
    for kind in ["rf", "svr"]:
        key = f"{optimizer}_{kind}"
        path = artifact_dir / key / "metrics.json"
        if not path.exists():
            raise FileNotFoundError(f"Hasil belum tersedia: {path}. Jalankan notebook optimasi terkait.")
        candidates[key] = json.loads(path.read_text(encoding="utf-8"))
comparison = pd.DataFrame(candidates).T.sort_values("cv_mse")
comparison[["optimizer", "model", "cv_rmse", "test_rmse", "test_mae", "test_r2"]]

## Latih seluruh kandidat dan simpan model terbaik

In [ ]:
def make_model(kind, params):
    if kind == "rf":
        return RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
    return Pipeline([("scale", StandardScaler()), ("svr", SVR(**params))])

X, y = data[FEATURE_COLS], data[TARGET_COL]
model_paths = {}
for key, item in candidates.items():
    kind = key.rsplit("_", 1)[1]
    model = make_model(kind, item["best_params"])
    model.fit(X, y)
    path = model_dir / f"{key}.joblib"
    joblib.dump(model, path)
    model_paths[key] = str(path.relative_to(root))

best_key = min(candidates, key=lambda key: candidates[key]["cv_mse"])
best_path = model_dir / f"{best_key}.joblib"
shutil.copy2(best_path, model_dir / "best_model.joblib")
summary = {
    "selected_model": best_key,
    "selection_metric": "cv_mse",
    "target": TARGET_COL,
    "features": FEATURE_COLS,
    "training_rows": len(data),
    "break_hour": break_hour,
    "best_params": candidates[best_key]["best_params"],
    "model_paths": model_paths,
}
(model_dir / "build_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

plot_df = comparison.sort_values("cv_rmse")
ax = plot_df["cv_rmse"].plot.bar(figsize=(8, 4), title="Perbandingan RMSE CV kandidat")
ax.set(xlabel="Kandidat", ylabel="RMSE CV")
plt.tight_layout()
plt.savefig(model_dir / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
summary